In [ ]:
import json
import os
import subprocess

environment = {**os.environ, "BUNDLE_VAR_api_key": "sample-api-key"}

def validate(folder):
    return subprocess.run(
        ["databricks", "bundle", "validate", "--output", "json"],
        cwd=folder, env=environment, capture_output=True, text=True,
    )

In [ ]:
for folder in ("01_explicit", "02_automatic"):
    result = validate(folder)
    if result.returncode:
        raise RuntimeError(result.stderr)
    variables = json.loads(result.stdout)["variables"]
    for name in ("api_key", "authorization"):
        if name in variables:
            value = variables[name]["value"]
            assert value == "[redacted]"
            print(folder, name, value)

In [ ]:
result = validate("03_rejected")
assert result.returncode != 0
assert 'Sensitive variable "api_key" is used in a non-sensitive field' in result.stderr
print(result.stderr.strip())